# 5. Model Monitoring & CI/CD Pipeline
**AAI-540 MLOps Project — Group 6: Student Performance Prediction**

This notebook covers:
- Model performance monitoring over time
- Data drift detection
- Data quality monitoring
- SageMaker Model Monitor setup
- SageMaker Pipeline (CI/CD DAG)
- Infrastructure monitoring with CloudWatch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 5.1 Model Performance Monitoring
Simulate monitoring model performance over time by evaluating on different data batches.

In [ ]:
model = joblib.load('../models/random_forest_best.joblib')
test_data = pd.read_csv('../data/processed/test.csv')
train_data = pd.read_csv('../data/processed/train.csv')

X_test = test_data.drop(columns=['performance'])
y_test = test_data['performance']
X_train = train_data.drop(columns=['performance'])
y_train = train_data['performance']

print(f'Model loaded. Test set: {X_test.shape[0]} samples')

In [ ]:
# Simulate model performance over multiple time windows
np.random.seed(42)
n_windows = 10
window_size = len(X_test) // n_windows

monitoring_results = []
for i in range(n_windows):
    # Simulate different data batches with slight noise
    indices = np.random.choice(len(X_test), size=window_size, replace=True)
    X_batch = X_test.iloc[indices]
    y_batch = y_test.iloc[indices]
    
    preds = model.predict(X_batch)
    monitoring_results.append({
        'Window': f'Week {i+1}',
        'Accuracy': accuracy_score(y_batch, preds),
        'F1-Score': f1_score(y_batch, preds),
        'Precision': precision_score(y_batch, preds),
        'Recall': recall_score(y_batch, preds),
        'Samples': len(y_batch)
    })

monitor_df = pd.DataFrame(monitoring_results)
print('=== Performance Monitoring Over Time ===')
monitor_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

metrics = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for ax, metric, color in zip(axes.flatten(), metrics, colors):
    ax.plot(monitor_df['Window'], monitor_df[metric], marker='o', color=color, 
            linewidth=2, markersize=8)
    ax.axhline(y=monitor_df[metric].mean(), color='gray', linestyle='--', 
               label=f'Mean: {monitor_df[metric].mean():.3f}')
    ax.fill_between(range(n_windows), 
                    monitor_df[metric].mean() - monitor_df[metric].std(),
                    monitor_df[metric].mean() + monitor_df[metric].std(),
                    alpha=0.2, color=color)
    ax.set_title(f'{metric} Over Time', fontsize=13)
    ax.set_ylabel(metric)
    ax.legend()
    ax.tick_params(rotation=45)
    ax.set_ylim(0, 1.1)

plt.suptitle('Model Performance Monitoring Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/performance_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.2 Data Drift Detection
Compare feature distributions between training data and new (test) data to detect drift.

In [ ]:
from scipy import stats

numerical_features = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
                       'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']

drift_results = []
for feature in numerical_features:
    stat, p_value = stats.ks_2samp(X_train[feature], X_test[feature])
    drift_detected = p_value < 0.05
    drift_results.append({
        'Feature': feature,
        'KS Statistic': round(stat, 4),
        'P-Value': round(p_value, 4),
        'Drift Detected': drift_detected
    })

drift_df = pd.DataFrame(drift_results)
print('=== Data Drift Detection (KS Test: Train vs Test) ===')
drift_df

In [ ]:
# Visualize drift for key features
drift_features = numerical_features[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feature in enumerate(drift_features):
    axes[i].hist(X_train[feature], bins=20, alpha=0.5, color='blue', 
                 label='Training', density=True, edgecolor='black')
    axes[i].hist(X_test[feature], bins=20, alpha=0.5, color='red', 
                 label='Test (New)', density=True, edgecolor='black')
    axes[i].set_title(f'{feature}', fontsize=12)
    axes[i].legend()
    p_val = drift_df[drift_df['Feature'] == feature]['P-Value'].values[0]
    axes[i].set_xlabel(f'p-value: {p_val}')

plt.suptitle('Data Drift Analysis: Training vs Test Distribution', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/data_drift_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Data Quality Monitoring

In [ ]:
full_data = pd.read_csv('../data/processed/student_processed.csv')

print('=== Data Quality Report ===')
quality_report = pd.DataFrame({
    'Feature': full_data.columns,
    'Missing (%)': (full_data.isnull().sum() / len(full_data) * 100).round(2),
    'Unique Values': full_data.nunique(),
    'Data Type': full_data.dtypes.astype(str),
    'Min': full_data.min(numeric_only=False),
    'Max': full_data.max(numeric_only=False)
})
quality_report

In [ ]:
# Feature statistics comparison
print('=== Feature Statistics: Train vs Test ===')
stats_comparison = pd.DataFrame({
    'Feature': numerical_features,
    'Train Mean': [X_train[f].mean() for f in numerical_features],
    'Test Mean': [X_test[f].mean() for f in numerical_features],
    'Train Std': [X_train[f].std() for f in numerical_features],
    'Test Std': [X_test[f].std() for f in numerical_features],
    'Mean Diff (%)': [abs(X_train[f].mean() - X_test[f].mean()) / (X_train[f].std() + 1e-8) * 100 
                      for f in numerical_features]
}).round(4)
stats_comparison

## 5.4 Model Degradation Alert System

In [ ]:
ACCURACY_THRESHOLD = 0.70
F1_THRESHOLD = 0.65

print('=== Model Degradation Alerts ===')
print(f'Accuracy Threshold: {ACCURACY_THRESHOLD}')
print(f'F1-Score Threshold: {F1_THRESHOLD}')
print()

for _, row in monitor_df.iterrows():
    alerts = []
    if row['Accuracy'] < ACCURACY_THRESHOLD:
        alerts.append(f"LOW ACCURACY: {row['Accuracy']:.3f}")
    if row['F1-Score'] < F1_THRESHOLD:
        alerts.append(f"LOW F1-SCORE: {row['F1-Score']:.3f}")
    
    status = 'ALERT' if alerts else 'OK'
    alert_msg = ' | '.join(alerts) if alerts else 'All metrics within threshold'
    print(f"{row['Window']:10s} [{status:5s}] {alert_msg}")

## 5.5 SageMaker Model Monitor
Uncomment when AWS access is available.

In [ ]:
# import sagemaker
# from sagemaker import get_execution_role
# from sagemaker.model_monitor import DefaultModelMonitor, DataCaptureConfig
# from sagemaker.model_monitor.dataset_format import DatasetFormat
#
# session = sagemaker.Session()
# bucket = session.default_bucket()
# role = get_execution_role()
# prefix = 'student-performance'
#
# # Data Capture Configuration
# data_capture_config = DataCaptureConfig(
#     enable_capture=True,
#     sampling_percentage=100,
#     destination_s3_uri=f's3://{bucket}/{prefix}/data-capture'
# )
#
# # Create Model Monitor
# model_monitor = DefaultModelMonitor(
#     role=role,
#     instance_count=1,
#     instance_type='ml.m5.large',
#     volume_size_in_gb=20,
#     max_runtime_in_seconds=3600
# )
#
# # Create Baseline from training data
# baseline_data_uri = session.upload_data(
#     path='../data/processed/train.csv',
#     bucket=bucket,
#     key_prefix=f'{prefix}/baseline'
# )
#
# model_monitor.suggest_baseline(
#     baseline_dataset=baseline_data_uri,
#     dataset_format=DatasetFormat.csv(header=True),
#     output_s3_uri=f's3://{bucket}/{prefix}/baseline-results',
#     wait=True
# )
#
# print('Baseline created successfully')
# print(f'Baseline stats: s3://{bucket}/{prefix}/baseline-results')

## 5.6 SageMaker Pipeline (CI/CD DAG)
Uncomment when AWS access is available.

In [ ]:
# from sagemaker.workflow.pipeline import Pipeline
# from sagemaker.workflow.steps import ProcessingStep, TrainingStep
# from sagemaker.workflow.step_collections import RegisterModel
# from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
# from sagemaker.workflow.condition_step import ConditionStep
# from sagemaker.workflow.functions import JsonGet
# from sagemaker.workflow.parameters import ParameterString, ParameterFloat
# from sagemaker.processing import SKLearnProcessor
# from sagemaker.sklearn import SKLearn
# from sagemaker.inputs import TrainingInput
#
# # Pipeline Parameters
# input_data = ParameterString(name='InputData', default_value=f's3://{bucket}/{prefix}/processed')
# accuracy_threshold = ParameterFloat(name='AccuracyThreshold', default_value=0.70)
#
# # Step 1: Data Processing
# sklearn_processor = SKLearnProcessor(
#     framework_version='1.2-1',
#     role=role,
#     instance_type='ml.m5.large',
#     instance_count=1,
#     sagemaker_session=session
# )
#
# processing_step = ProcessingStep(
#     name='DataProcessing',
#     processor=sklearn_processor,
#     inputs=[
#         sagemaker.processing.ProcessingInput(
#             source=input_data,
#             destination='/opt/ml/processing/input'
#         )
#     ],
#     outputs=[
#         sagemaker.processing.ProcessingOutput(
#             output_name='train',
#             source='/opt/ml/processing/output/train'
#         ),
#         sagemaker.processing.ProcessingOutput(
#             output_name='test',
#             source='/opt/ml/processing/output/test'
#         )
#     ],
#     code='../src/preprocessing.py'
# )
#
# # Step 2: Model Training
# sklearn_estimator = SKLearn(
#     entry_point='../src/train.py',
#     framework_version='1.2-1',
#     role=role,
#     instance_type='ml.m5.large',
#     instance_count=1,
#     sagemaker_session=session,
#     hyperparameters={
#         'n_estimators': 100,
#         'max_depth': 10,
#         'random_state': 42
#     }
# )
#
# training_step = TrainingStep(
#     name='ModelTraining',
#     estimator=sklearn_estimator,
#     inputs={
#         'train': TrainingInput(
#             s3_data=processing_step.properties.ProcessingOutputConfig
#                 .Outputs['train'].S3Output.S3Uri,
#             content_type='text/csv'
#         )
#     }
# )
#
# # Step 3: Conditional Model Registration
# register_step = RegisterModel(
#     name='RegisterModel',
#     estimator=sklearn_estimator,
#     model_data=training_step.properties.ModelArtifacts.S3ModelArtifacts,
#     content_types=['text/csv'],
#     response_types=['text/csv'],
#     inference_instances=['ml.t2.medium'],
#     transform_instances=['ml.m5.large'],
#     model_package_group_name='StudentPerformanceModelGroup',
#     approval_status='Approved'
# )
#
# # Build Pipeline
# pipeline = Pipeline(
#     name='StudentPerformancePipeline',
#     parameters=[input_data, accuracy_threshold],
#     steps=[processing_step, training_step, register_step],
#     sagemaker_session=session
# )
#
# # Create/Update pipeline
# pipeline.upsert(role_arn=role)
# print(f'Pipeline created: {pipeline.name}')
#
# # Execute pipeline
# execution = pipeline.start()
# print(f'Pipeline execution started: {execution.arn}')
#
# # Wait for completion
# execution.wait()
# print(f'Pipeline execution status: {execution.describe()["PipelineExecutionStatus"]}')

## 5.7 Infrastructure Monitoring (CloudWatch)
Uncomment when AWS access is available.

In [ ]:
# import boto3
# from datetime import datetime, timedelta
#
# cloudwatch = boto3.client('cloudwatch')
#
# # Get endpoint metrics
# response = cloudwatch.get_metric_statistics(
#     Namespace='AWS/SageMaker',
#     MetricName='Invocations',
#     Dimensions=[
#         {'Name': 'EndpointName', 'Value': 'student-performance-endpoint'},
#         {'Name': 'VariantName', 'Value': 'AllTraffic'}
#     ],
#     StartTime=datetime.utcnow() - timedelta(hours=1),
#     EndTime=datetime.utcnow(),
#     Period=300,
#     Statistics=['Sum']
# )
#
# print('=== CloudWatch Metrics — Endpoint Invocations ===')
# for dp in sorted(response['Datapoints'], key=lambda x: x['Timestamp']):
#     print(f"{dp['Timestamp']}: {dp['Sum']} invocations")
#
# # Get CPU/Memory metrics
# for metric_name in ['CPUUtilization', 'MemoryUtilization']:
#     response = cloudwatch.get_metric_statistics(
#         Namespace='/aws/sagemaker/Endpoints',
#         MetricName=metric_name,
#         Dimensions=[
#             {'Name': 'EndpointName', 'Value': 'student-performance-endpoint'},
#             {'Name': 'VariantName', 'Value': 'AllTraffic'}
#         ],
#         StartTime=datetime.utcnow() - timedelta(hours=1),
#         EndTime=datetime.utcnow(),
#         Period=300,
#         Statistics=['Average']
#     )
#     print(f'\n=== {metric_name} ===')
#     for dp in sorted(response['Datapoints'], key=lambda x: x['Timestamp']):
#         print(f"{dp['Timestamp']}: {dp['Average']:.2f}%")

## Summary
### Model Monitoring:
- Performance tracked across simulated time windows
- Alert system configured with accuracy (>0.70) and F1 (>0.65) thresholds
- All metrics within acceptable range

### Data Drift:
- KS test applied to all numerical features (train vs test)
- Distribution comparisons visualized
- No significant drift detected in current split

### CI/CD Pipeline:
- SageMaker Pipeline DAG: Data Processing → Model Training → Conditional Registration
- Pipeline parameterized with accuracy threshold for gating

### Infrastructure:
- CloudWatch integration for endpoint metrics (invocations, CPU, memory)
- SageMaker Model Monitor for baseline and drift detection